# Day-Ahead Solar PV Forecasting — Results

Forecasting Belgian grid-scale photovoltaic output 24 hours ahead from numerical
weather predictions, with calibrated uncertainty bands.

The benchmark throughout is **Elia's own operational day-ahead forecast**, which ships
in the same dataset. Every number here is measured against what a transmission system
operator actually runs in production.

**Headline result:** Ridge reaches parity with Elia (−0.14% skill). Nothing beats it.

---

This notebook is a *read-only consumer* of the pipeline in `src/`. It loads finished
artifacts and presents them — it contains no pipeline logic of its own. To regenerate
the underlying data and models:

```
python -m src.data.build_dataset
python -m src.features.build_features
python -m src.models.tune
python -m src.models.train
python -m src.models.quantile
```

In [ ]:
import sys
from pathlib import Path

# The notebook lives in notebooks/; the package lives one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluation.metrics import score, score_intervals
from src.features.build_features import FEATURE_COLS, TARGET_COL
from src.preprocessing.split import chronological_split

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = ROOT / "data" / "dayahead"
MODELS = ROOT / "models" / "dayahead"
print("root:", ROOT)

## 1. Data

Two sources, joined on UTC timestamp.

| Source | Provides | Cadence |
|---|---|---|
| **Elia ODS032** | Measured Belgian PV output, Elia's own day-ahead forecast, P10/P90 bands, installed capacity | 15 min |
| **Open-Meteo previous-runs** | Archived NWP at a fixed 1-day lead — irradiance, temperature, cloud, wind | hourly |

The weather columns are *archived forecasts*, not observations. That is what makes the
day-ahead framing leak-free: the model only ever sees what was genuinely knowable the
day before.

Weather is sampled at **all eleven Belgian provinces** and averaged by each one's
installed PV capacity — cloud over Antwerp matters in proportion to how many panels sit
in Antwerp.

In [ ]:
features = pd.read_parquet(DATA / "features.parquet")
joined = pd.read_parquet(DATA / "belgium_hourly.parquet")

print(f"rows        : {len(features):,}")
print(f"range       : {features.index.min()}  ->  {features.index.max()}")
print(f"features    : {len(FEATURE_COLS)}")
print(f"daytime rows: {int((features['is_day'] == 1).sum()):,} "
      f"({100 * (features['is_day'] == 1).mean():.1f}%)")
features[["measured", "dayaheadforecast", "monitoredcapacity", "cf"]].describe().T

### Integrity

The previous version of this project used a dataset with 23.8% of its timestamps
missing and sensor readings of −7999 °C. Everything now gets checked before use.

In [ ]:
expected = pd.date_range(joined.index.min(), joined.index.max(), freq="1h", tz="UTC")
missing = expected.difference(joined.index)

print(f"expected hours : {len(expected):,}")
print(f"present        : {len(joined):,}")
print(f"missing        : {len(missing):,} ({100 * len(missing) / len(expected):.3f}%)")
print(f"duplicates     : {joined.index.duplicated().sum()}")
print(f"nulls          : {int(joined.isnull().sum().sum())}")

## 2. What we are predicting

Output is modelled as **capacity factor** — megawatts divided by installed capacity —
rather than raw megawatts. The Belgian fleet grew from 8.8 to 12.1 GW across this
window, so in raw MW the model cannot distinguish "sunnier day" from "more panels".

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

daily = features["measured"].resample("1D").mean()
axes[0].plot(daily.index, daily.values, lw=0.8, color="#d95f02")
axes[0].set_ylabel("MW (daily mean)")
axes[0].set_title("Measured output — raw megawatts")

axes[1].plot(features.index, features["monitoredcapacity"], lw=1.2, color="#1b9e77")
axes[1].set_ylabel("MW installed")
axes[1].set_title("Installed capacity — why raw MW is misleading")
plt.tight_layout()

In [ ]:
day = features[features["is_day"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

by_hour = day.groupby(day.index.hour)["cf"].mean()
axes[0].plot(by_hour.index, by_hour.values, marker="o", color="#7570b3")
axes[0].set(xlabel="hour (UTC)", ylabel="mean capacity factor", title="Diurnal profile")

by_month = day.groupby(day.index.month)["cf"].mean()
axes[1].bar(by_month.index, by_month.values, color="#e7298a", alpha=0.8)
axes[1].set(xlabel="month", ylabel="mean capacity factor", title="Seasonal profile")
plt.tight_layout()

## 3. A bug worth showing

Open-Meteo stamps radiation with the **end** of the hour it covers; Elia stamps hourly
means with the **start**. Joining them naively offsets sunshine from power by a full
hour.

It was caught by scanning candidate shifts for peak correlation, using Elia's own
forecast as a control — the control must peak at zero, or the join itself is wrong.

In [ ]:
cf = features["cf"]
shifts = range(-3, 4)

ghi_corr = [cf.corr(features["shortwave_radiation"].shift(s)) for s in shifts]
ref_corr = [cf.corr(features["elia_dayahead_cf"].shift(s)) for s in shifts]

plt.figure()
plt.plot(list(shifts), ghi_corr, marker="o", label="forecast GHI")
plt.plot(list(shifts), ref_corr, marker="s", label="Elia forecast (control)")
plt.axvline(0, color="grey", ls="--", lw=1)
plt.xlabel("shift applied (hours)")
plt.ylabel("correlation with capacity factor")
plt.title("Alignment check — both must peak at zero")
plt.legend()

print(f"GHI peaks at shift    : {list(shifts)[int(np.argmax(ghi_corr))]:+d} h")
print(f"control peaks at shift: {list(shifts)[int(np.argmax(ref_corr))]:+d} h")

## 4. Features

28 features in four families. Nothing here uses information unavailable at forecast
issue time.

- **Weather forecast** (10) — irradiance components, temperature, cloud, wind,
  precipitation, plus the spread of irradiance *between provinces*
- **Solar geometry** (7) — sun elevation/azimuth and the clear-sky ceiling. Pure
  astronomy, knowable years ahead
- **Calendar** (4) — cyclic hour and day-of-year encodings
- **Physics** (2) — estimated cell temperature and a first-principles yield estimate
- **History** (4) — lagged output at 72/96/168 h, plus a 7-day rolling mean
- **Baseline** (1) — Elia's own forecast

Lags start at **72 hours**, not 24. Two constraints force this: Elia issues the forecast
the morning before (so newer actuals would leak), and the data feeds leave a ~24 h blind
spot (so newer actuals are not even available at serving time).

In [ ]:
for name, cols in [
    ("weather",  [c for c in FEATURE_COLS if "radiation" in c or "cloud" in c
                  or "temperature_2m" in c or "humidity" in c or "wind" in c
                  or "precipitation" in c or "dispersion" in c]),
    ("solar",    [c for c in FEATURE_COLS if "solar_" in c or "clearsky" in c]),
    ("calendar", [c for c in FEATURE_COLS if "_sin" in c or "_cos" in c]),
    ("physics",  ["cell_temp_est", "physical_yield"]),
    ("history",  [c for c in FEATURE_COLS if c.startswith("cf_")]),
    ("baseline", ["elia_dayahead_cf"]),
]:
    print(f"{name:<9} ({len(cols):>2}) {', '.join(cols)}")

### The clear-sky ceiling

The single most useful engineered feature is the ratio of forecast irradiance to what a
**cloudless** sky would deliver. It converts "794 W/m²" into "how cloudy will it be" —
a number meaning the same thing in December as in June.

In [ ]:
sample = features.loc["2026-06-10":"2026-06-17"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(sample.index, sample["clearsky_ghi"], alpha=0.25,
                color="#fdae61", label="clear-sky ceiling")
ax.plot(sample.index, sample["shortwave_radiation"], lw=1.4,
        color="#d7191c", label="forecast GHI")
ax.set_ylabel("W/m²")
ax.set_title("Forecast irradiance against the cloudless-sky ceiling — one week")
ax.legend(loc="upper right")
plt.tight_layout()

## 5. Splits

Strictly chronological — never random. Solar is seasonal, so a random split would let
the model see July while being tested on July.

The test period is a **full 12 months** so the headline covers every season rather than
a flattering stretch of summer.

In [ ]:
train, val, test = chronological_split(features)

for name, part in [("train", train), ("val", val), ("test", test)]:
    day_part = part[part["is_day"] == 1]
    print(f"{name:<6} {len(part):>7,} rows  ({len(day_part):>6,} daytime)  "
          f"{part.index.min():%Y-%m-%d} -> {part.index.max():%Y-%m-%d}")

## 6. Results

Daylight hours only. Including night would add thousands of trivially correct zeros and
inflate every score — roughly half of all rows are dark.

Errors are percentages of installed capacity, the standard normalisation in solar
forecasting.

In [ ]:
test_day = test[test["is_day"] == 1]
X_test, y_test = test_day[FEATURE_COLS], test_day[TARGET_COL]

ridge_bundle = joblib.load(MODELS / "ridge.pkl")
lgbm_bundle = joblib.load(MODELS / "lgbm.pkl")

ridge_pred = pd.Series(
    ridge_bundle["model"].predict(
        ridge_bundle["scaler"].transform(X_test)).clip(0, 1), index=y_test.index)
lgbm_pred = pd.Series(
    lgbm_bundle["model"].predict(X_test).clip(0, 1), index=y_test.index)

results = pd.DataFrame([
    score(y_test, test_day["elia_dayahead_cf"], label="Elia day-ahead"),
    score(y_test, ridge_pred, label="Ridge"),
    score(y_test, lgbm_pred, label="LightGBM"),
]).set_index("label")

base = results.loc["Elia day-ahead", "nRMSE"]
results["skill_vs_elia_%"] = 100 * (1 - results["nRMSE"] / base)
results.round(3)

**Ridge wins, and the simpler model beating the more complex one is the finding, not an
accident.** This task is essentially a near-linear correction to Elia's forecast — which
is what Ridge does natively and what a tree ensemble approximates badly with
axis-aligned splits.

In [ ]:
week = slice("2026-06-10", "2026-06-17")
actual = (test_day.loc[week, "cf"] * test_day.loc[week, "monitoredcapacity"])
pred = (ridge_pred.loc[week] * test_day.loc[week, "monitoredcapacity"])
elia = test_day.loc[week, "dayaheadforecast"]

plt.figure(figsize=(12, 4))
plt.plot(actual.index, actual.values, lw=2, color="black", label="actual")
plt.plot(pred.index, pred.values, lw=1.4, ls="--", color="#1f78b4", label="Ridge")
plt.plot(elia.index, elia.values, lw=1.4, ls=":", color="#e31a1c", label="Elia")
plt.ylabel("MW")
plt.title("One week of the test year")
plt.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ax, (pred, name) in zip(axes, [(ridge_pred, "Ridge"), (lgbm_pred, "LightGBM")]):
    ax.scatter(y_test, pred, s=3, alpha=0.15, color="#1f78b4")
    ax.plot([0, 0.8], [0, 0.8], color="red", lw=1)
    ax.set(xlabel="actual capacity factor", ylabel="predicted", title=name)
plt.tight_layout()

### Skill by quarter

An aggregate improvement can hide a model that wins in one season and loses in another —
worthless operationally, since you cannot choose which months to run.

In [ ]:
quarters = test_day.index.tz_convert(None).to_period("Q")
rows = []
for q in sorted(set(quarters)):
    m = quarters == q
    base_rmse = np.sqrt(((y_test[m] - test_day.loc[m, "elia_dayahead_cf"]) ** 2).mean())
    rows.append({
        "quarter": str(q),
        "n": int(m.sum()),
        "Ridge": 100 * (1 - np.sqrt(((y_test[m] - ridge_pred[m]) ** 2).mean()) / base_rmse),
        "LightGBM": 100 * (1 - np.sqrt(((y_test[m] - lgbm_pred[m]) ** 2).mean()) / base_rmse),
    })

quarterly = pd.DataFrame(rows).set_index("quarter")
ax = quarterly[["Ridge", "LightGBM"]].plot(kind="bar", figsize=(10, 3.8),
                                           color=["#1f78b4", "#33a02c"])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("skill vs Elia (%)")
ax.set_title("Positive = better than Elia")
plt.tight_layout()
quarterly.round(2)

## 7. Why cross-validation and the test year disagreed

Cross-validation said LightGBM beat Elia by +2.97%. The test year said it lost by 3.55%.
That gap is not ordinary overfitting — **Elia's own forecast improved between the two
periods**, so a model trained to correct their 2024 residuals was fixing mistakes they
had since stopped making.

In [ ]:
day_all = features[features["is_day"] == 1].copy()
day_all["quarter"] = day_all.index.tz_convert(None).to_period("Q")

elia_err = day_all.groupby("quarter", observed=True).apply(
    lambda g: 100 * np.sqrt(((g["cf"] - g["elia_dayahead_cf"]) ** 2).mean()),
    include_groups=False)

ax = elia_err.plot(kind="bar", figsize=(10, 3.8), color="#e31a1c", alpha=0.8)
ax.axvline(5.5, color="black", ls="--", lw=1.2)
ax.text(5.6, elia_err.max() * 0.95, "test year begins", fontsize=9)
ax.set_ylabel("Elia nRMSE (% of capacity)")
ax.set_title("Elia's forecast is a moving target — it keeps getting better")
plt.tight_layout()

print("Year-on-year, same quarter:")
for a, b in [("2025Q1", "2026Q1"), ("2025Q2", "2026Q2"), ("2024Q4", "2025Q4")]:
    if a in elia_err.index.astype(str) and b in elia_err.index.astype(str):
        va = elia_err[elia_err.index.astype(str) == a].iloc[0]
        vb = elia_err[elia_err.index.astype(str) == b].iloc[0]
        print(f"  {a} {va:.3f}%  ->  {b} {vb:.3f}%   ({100 * (vb / va - 1):+.1f}%)")

## 8. Uncertainty

A single number is not actionable. Whoever schedules reserve capacity needs to know how
wrong the forecast might be, so three quantile models produce a P10/P50/P90 band.

Probabilistic forecasts need **two** metrics, never one alone:
- **Coverage** — does the band contain the truth as often as it claims? (target 80%)
- **Sharpness** — is it narrow enough to be useful? Any model can hit perfect coverage
  by predicting "somewhere between zero and maximum"

In [ ]:
qbundle = joblib.load(MODELS / "lgbm_quantile.pkl")

preds = {name: model.predict(X_test).clip(0, 1)
         for name, model in qbundle["models"].items()}
stacked = np.sort(np.column_stack([preds["P10"], preds["P50"], preds["P90"]]), axis=1)
lower, median, upper = (pd.Series(stacked[:, i], index=y_test.index) for i in range(3))

pd.DataFrame([
    score_intervals(y_test, lower, median, upper, label="LightGBM quantile"),
    score_intervals(y_test,
                    test_day["dayaheadconfidence10"] / test_day["monitoredcapacity"],
                    test_day["elia_dayahead_cf"],
                    test_day["dayaheadconfidence90"] / test_day["monitoredcapacity"],
                    label="Elia band"),
]).set_index("label").round(3)

In [ ]:
cap = test_day.loc[week, "monitoredcapacity"]

plt.figure(figsize=(12, 4))
plt.fill_between(cap.index, lower.loc[week] * cap, upper.loc[week] * cap,
                 alpha=0.3, color="#1f78b4", label="P10-P90")
plt.plot(cap.index, median.loc[week] * cap, lw=1.4, color="#1f78b4", label="P50")
plt.plot(cap.index, test_day.loc[week, "cf"] * cap, lw=2, color="black", label="actual")
plt.ylabel("MW")
plt.title("Forecast band against outcome — one week")
plt.legend()
plt.tight_layout()

## 9. Conclusions

**Ridge reaches parity with Elia (−0.14% skill); nothing beats it.** Matching a
transmission operator's production forecast using free public weather data is a
reasonable outcome. Claiming to beat it would not be supported by these numbers.

What the build surfaced along the way:

1. **Geography beat modelling.** Moving from one grid point at the national centroid to
   eleven capacity-weighted provincial points lifted correlation from 0.938 to 0.958 and
   closed most of the gap to Elia. It was the largest single improvement in the project.

2. **The benchmark is a moving target.** Elia's error fell ~10% between development and
   test — 19% year-on-year in Q1. Static correction models go stale.

3. **Retraining helps less than expected.** Walk-forward backtesting found an expanding
   window beats a trailing 12-month one. With this little data, discarding history to
   chase recency costs more than staleness does.

4. **Simpler won.** Ridge beat a tuned gradient-boosting ensemble.

### Limitations

- ~7,500 daylight training rows is thin. Open-Meteo's Historical Forecast API reaches
  back to 2021 instead of 2024 and would roughly double this — the largest remaining
  lever.
- Grid-aggregate data, so no module temperature or DC power; cell temperature is
  estimated from air temperature and irradiance.
- Provinces are sampled at their capitals, a proxy for each one's true
  capacity-weighted centroid.
- The quantile models reuse regularisation tuned for squared error rather than pinball
  loss, and did not improve when the point forecast did.
- **The test year was evaluated four times** as the pipeline changed. Each change was
  selected on cross-validation, not test performance, but repeated looks still bias the
  estimate optimistically. Read −0.14% as "roughly parity", not as a precise
  measurement.